In [1]:
import sys
import os

sys.path.append('/scratch2/mrenaudin/colorlessgreenRNNs')

In [2]:
from src.language_models import model as m
import torch
from utils import NounPPDataset, collate_fn_nounpp
from src.language_models.dictionary_corpus import Dictionary
from torch.utils.data import DataLoader
from collections import defaultdict



/home/mrenaudin/.conda/envs/leaps3/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
device = torch.device('cpu')
model = m.CBR_RNN(50001, 650, 650, 1, 0, device)
checkpoint = torch.load('/scratch2/mrenaudin/colorlessgreenRNNs/checkpoints/test_attention_650/epoch_38.pt', map_location='cpu')
data_path = "/scratch2/mrenaudin/colorlessgreenRNNs/english_data"
dictionary = Dictionary(data_path)
nounpp = "//scratch2/mrenaudin/colorlessgreenRNNs/NounPP/Stimuli/nounpp.txt"


In [4]:
test_dataset = NounPPDataset(nounpp, dictionary)
test_dataloader = DataLoader(test_dataset, batch_size=1024, collate_fn=collate_fn_nounpp)

In [5]:
model.load_state_dict(checkpoint['model_state_dict'])

<All keys matched successfully>

In [6]:
temp = checkpoint['epoch'] #that's because of mistake in save checkpoints function

In [11]:
checkpoint

{'epoch': 0.1188502227437019,
 'model_state_dict': OrderedDict([('encoder.weight',
               tensor([[ 0.3335,  0.5166, -0.0126,  ...,  0.2925, -0.0226,  0.3091],
                       [-0.3646, -0.1746, -0.2422,  ...,  0.3091,  0.5116, -0.4240],
                       [ 0.1298, -0.0304,  0.0236,  ...,  0.0506, -0.0707,  0.0488],
                       ...,
                       [ 0.0533, -0.3885,  0.0872,  ...,  0.0202, -0.0694, -0.0145],
                       [ 0.3967,  0.2758, -0.0341,  ...,  0.3553,  0.3348,  0.2553],
                       [ 0.0515,  0.0654, -0.0585,  ...,  0.2444,  0.2960,  0.0729]])),
              ('q.weight',
               tensor([[-0.0013,  0.0835, -0.1486,  ...,  0.0772, -0.2472, -0.2034],
                       [ 0.2643, -0.1248,  0.0695,  ...,  0.3107, -0.2703, -0.0039],
                       [ 0.0836,  0.0824, -0.0921,  ..., -0.0542, -0.0574,  0.0130],
                       ...,
                       [ 0.0850,  0.0392,  0.0381,  ...,  0.3007, 

In [9]:
def eval(model, test_dataloader, temperature):
    condition_accuracies = defaultdict(int)
    condition_counts = defaultdict(int)
    correct_pred = 0
    sentence_details = []
    model.eval()
    # Forward pass with hidden state update word by word
    with torch.no_grad():
        for batch in test_dataloader:
            out = None
            written = batch["sentence"]
            sentence = batch["encoded_sentence"]
            correct = batch["encoded_correct"]
            wrong = batch["encoded_wrong"]
            condition = batch["condition"]
            batch_size = sentence.size(0)

            sent = sentence[:, :5].transpose(0, 1)
            cache = model.init_cache(sent,1)  # regarder si on peut mettre du priming
            # for i in range(sent.shape[1]):
            out, cache = model(sent, cache, 1, temperature, True)
            log_probs = torch.nn.functional.log_softmax(
                out, dim=-1
            )  # s(out.squeeze(0))
            # déja sur correct et wrong log probs, pas les même résultats que sur extract_predictions.py
            correct_log_probs = log_probs[
                -1, torch.arange(batch_size), correct
            ]  # Shape: [512]
            wrong_log_probs = log_probs[-1, torch.arange(batch_size), wrong]
            correct_predictions = correct_log_probs >= wrong_log_probs

            for i in range(batch_size):
                cond = condition[i]
                pred = correct_predictions[i].item()  # Convert tensor to Python boolean
                condition_counts[cond] += 1
                condition_accuracies[cond] += pred

                sentence_details.append(
                    {
                        "sentence": written[i],
                        "condition": condition[i],
                        "correct_log_prob": correct_log_probs[i],
                        "wrong_log_prob": wrong_log_probs[i],
                        "model_prefers_correct": pred,
                    }
                )

    final_accuracies = {
        cond: condition_accuracies[cond] / condition_counts[cond]
        for cond in condition_accuracies
    }
    return final_accuracies

In [10]:
eval(model, test_dataloader, temp)

{'singular singular': 0.87,
 'singular plural': 0.499,
 'plural singular': 0.855,
 'plural plural': 0.828}